# M7L2 - DAP-Seq: Interpreting Results

After sequencing, the raw data must be processed to identify binding sites and discover motifs. We will cover the analysis steps of DAP-Seq within the next lab, but let's focus on interpreting the results of DAP-Seq experiments!


## Introduction: 
When a DAP-Seq experiment is complete and the data have been processed, researchers are presented with several types of output files and visualizations. In this section, we will explore the key components of DAP-Seq results using examples from the paper: [Cistrome and Epicistrome Features Shape the
Regulatory DNA Landscape](http://dx.doi.org/10.1016/j.cell.2016.04.038), focusing on what the data looks like and what it tells us about TF biology.

The primary outputs from a DAP-Seq experiment include:
- Peak files that define genomic regions where the transcription factor binds
- Sequence motifs that represent the preferred binding sequence
- Genome browser tracks that visualize binding across chromosomes
- Various quality metrics that indicate whether the experiment was successful.


Each of these components provides complementary information about transcription factor behavior and regulatory network architecture.

# Peak Files - Where Transcription Factors Bind
The most fundamental output from DAP-Seq is a list of peaks. **Peaks are genomic coordinates where the transcription factor shows significant enrichment of binding.** These peaks are typically provided in `BED` format (Browser Extensible Data), a standard file format in genomics that specifies chromosome, start position, end position, and additional information about each binding site.

A typical peak file entry might look like this:
> `Chr1    1547200    1547450    peak_1    856    .    45.2    85.6    -1    125`

This line indicates that on Chromosome 1, there is a peak spanning from position 1,547,200 to 1,547,450 base pairs. The subsequent columns provide metadata: 

- a peak identifier
- a score reflecting binding strength
- strand information
- fold enrichment over background
- statistical significance, and the precise summit location within the peak region.

Understanding these metrics helps researchers prioritize which peaks are most likely to represent biologically important binding events.

The number of peaks identified varies dramatically between different transcription factors. The strength of a peak, measured by metrics like fold enrichment and read depth, correlates with binding affinity and likely biological importance. Strong peaks with 50-fold or 100-fold enrichment over background typically represent high-affinity binding sites where the TF binds tightly and consistently. These sites are most likely to drive gene regulation under normal cellular conditions. Weaker peaks with lower fold enrichment might represent lower-affinity sites that only become functionally important under specific conditions or in combination with other regulatory factors.

**The Fraction of Reads in Peaks (FRiP) is a critical quality metric that indicates what percentage of all sequenced reads fall within the called peaks.** - In a successful DAP-Seq experiment, FRiP values typically range from 10% to 60%. 
- A FRiP below 5% suggests the experiment may have failed or the transcription factor binds very weakly to DNA.

To examine peaks, we can use a genome browser like IGV: 

*[Third-party figure omitted from the public version; original source not confirmed.]*

Learn more about `BED` format and peak calling with `MACS2` here: [Peak Calling and BED format - BioCore](https://biocorecrg.github.io/PhD_course_genomics_format_2021/peak_calling.html)

#### Spatial Distribution of Peaks

Examining where peaks are located relative to genes provides immediate biological insight. Most transcription factors show enrichment in gene promoters (the region immediately upstream of transcription start sites), which makes sense given that TFs regulate transcription initiation. However, the precise distribution pattern varies by TF family and function.

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*

For example, the bZIP transcription factor ABI5, which regulates abscisic acid responses, shows strong enrichment in promoter regions and 5' untranslated regions (5' UTRs). This distribution is consistent with ABI5's role as a transcriptional activator that must bind near transcription start sites to recruit RNA polymerase and initiation factors. 


# Using controls to eliminate background signals (Noise)

When analyzing DAP-Seq, ChIP-Seq, or ATAC-Seq data, one of the fundamental challenges is distinguishing true transcription factor binding events (or chromatin accessibility) from technical noise and non-specific background signal. These background reads arise from multiple sources including random DNA fragmentation, non-specific DNA binding during immunoprecipitation or affinity purification, sequencing errors, PCR amplification bias, and mappability issues, where some genomic regions are inherently easier to sequence than others.

Simply, we want to identify regions where read density is enriched significantly above what would be expected by chance. We use background modeling in our DAP-Seq experiments to avoid: 
- **False Positives:** occur when random fluctuations in the background signal are mistakenly called peaks
- **False Negatives:** true binding sites are dismissed because they don't rise sufficiently above a noisy background

Different experimental designs(ChIP vs DAP) use different types of controls to model the background signal. IN DAP-Seq, an input DNA control would be the adaptor-ligated genomic DNA library before mixing it with the transcription factor. Some peak callers, particularly `MACS2`, can operate without an explicit control sample by modeling the expected background based on the ChIP or DAP-Seq sample itself. The algorithm assumes that most of the genome is not bound by the TF, so the baseline read distribution across most regions represents background.

# Sequence Motifs

Beyond knowing the location of TF binding, the sequence that the TF binds to is critical for predicting additional binding sites and understanding the regulation logic. DAP-Seq motif discovery algorithms analyze the DNA sequences at peak summits to identify overrepresented patterns: a binding motif. 

### Position Weight Matrices

The motif is mathematically represented as a Position Weight Matrix (PWM), which captures the probability of finding each nucleotide (A, C, G, or T) at each position in the binding site. The PWM is most commonly visualized as a sequence logo, where each position in the motif is represented by stacked letters whose heights indicate information content. The width of the motif (number of informative positions) varies by transcription factor family. The bZIP family, which includes ABI5, typically produces motifs 8-12 base pairs wide. The ARF family produces shorter motifs around 6 base pairs centered on TGTC. The distribution of motifs within peaks also matters. High-quality motifs should show "central enrichment."

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*


For example, the ABI5 motif PWM might show that position 4 is almost always a C (probability ~0.95), position 5 is almost always an A (probability ~0.93), and position 6 is almost always a C (probability ~0.94), forming the core CACGTG sequence known as a G-box. The PWM allows for some flexibility—not every ABI5 binding site needs to be a perfect CACGTG. Position 1 might show equal probabilities of G and A (both ~0.4), indicating either nucleotide is tolerated. **As David said in his lecture, biological systems are meant to be reversible and modifiable. This means that these TFs recognize familys of related motifs, not just a single motif.**


### Comparing DAP-Seq motifs to Known Motifs

One way to validate DAP-Seq results is by comparing discovered motifs to previously published motifs from databases like JASPAR, TRANSFAC, or CIS-BP (which contains PBM-derived motifs). For ABI5, the DAP-Seq motif matches the known G-box motif, confirming the experiment worked correctly. However, the DAP-Seq motif is often longer and contains more information than PBM-derived motifs for the same TF.

### Motif Quality and Central Enrichment
Not every peak necessarily contains a clear motif. In successful DAP-Seq experiments, 70-90% of peaks contain at least one instance of the discovered motif. Peaks without motifs might represent false positives from the peak calling algorithm, or they might indicate the TF was recruited to DNA indirectly (This is less likely in the chromatin-free and single TF context of DAP-Seq).


# Genome Browser Visualization

Genome browsers like IGV (Integrative Genomics Viewer) or UCSC Genome Browser display DAP-Seq peaks as tracks aligned with gene annotations, allowing us to immediately see which genes are likely regulated by the transcription factor.

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*

A typical genome browser view shows multiple aligned tracks. The top track might display gene structure with exons as thick blocks and introns as thin lines, with arrows indicating transcription direction. Below this, the DAP-Seq track appears as a histogram or peak visualization showing read coverage along the chromosome. Where the transcription factor binds strongly, the track shows tall peaks. These regions without binding remain at baseline.


## Read Density Heatmaps

A common visualization is the read density heatmap centered on transcription start sites (TSS). Each row in the heatmap represents one gene, with the TSS at the center (position 0). Color intensity indicates DAP-Seq signal strength, with hot colors (red) representing high read depth and cool colors (yellow/white) representing low or no signal. Genes (rows) are often sorted by signal intensity, creating a visual gradient from strong binding to no binding.

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038), Fig. 6E.]*

For ARF5, O'Malley et al. created such a heatmap comparing target genes to non-target genes (Figure 6E). The target genes show a strong red signal concentrated in the promoter region (just upstream of the TSS), while background genes show minimal signal. Strikingly, the target genes show a periodic pattern with signal peaks occurring every ~300 base pairs upstream of the TSS, reflecting the phased arrangement of ARF binding sites in these promoters.

This phasing pattern at ~300 bp periodicity is close to nucleosome spacing in vivo (nucleosomes occupy ~147 bp of DNA and are separated by ~20-80 bp linkers). The heatmap suggests ARF5 binding sites in target gene promoters are positioned to coincide with nucleosome-depleted regions that exist in active promoters. 


## Motif Similarity Trees

When analyzing many transcription factors together, clustering motifs by similarity reveals evolutionary and functional relationships. When multiple unrelated TFs share the same motif, it suggests potential regulatory redundancy or cooperative binding.  Motif similarity can be quantified using algorithms that compare Position Weight Matrices, such as TOMTOM or the approach used in the MotIV package. These algorithms calculate similarity scores between every pair of motifs, which can then be visualized as a hierarchical clustering tree (dendrogram). In this tree, TFs with very similar motifs cluster together on nearby branches, while TFs with dissimilar motifs are separated on distant branches.

For the bZIP family, the paper showed that the motif similarity tree closely matches the phylogenetic tree based on protein sequences (Figure 3A). This means that closely related bZIP proteins have conserved their DNA-binding specificity through evolution: 

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038), Fig. 3A.]*

 For example, TGA1, TGA2, TGA3, and TGA5 form a tight cluster with nearly identical TGACG-core motifs, reflecting their common ancestry and overlapping regulatory roles.

## Target Gene prediction / Functional analysis (GO Enrichment)

Through the use of our peak data, our motifs, and Functional analysis with GO terms, we can predict which genes each transcription factor regulates, and then determine what biological processes those genes control. The first step is associating each peak with its likely target genes. Then, we use Gene Ontology (GO) enrichment analysis to test whether the target genes are enriched for specific biological functions. After this, we can cluster the GO terms for these genes and see if our TFs, or familys of TFs that share motifs, target genes with similar functions. 

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*

In the paper, 57 representative TFs were analyzed in detail, with each transcription factor associated with hundreds to thousands of predicted target genes based on DAP-Seq peaks. ARF5 peaks predicted ~3,000 target genes, consistent with its role as a master regulator of development. Then, ARF5's ~3,000 predicted target genes to GO enrichment analysis returns highly significant enrichment for terms like "auxin-activated signaling pathway," "regulation of growth," "shoot system development," and "cell differentiation", confirming the role of ARF5 in auxin response during development. 


*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038), Fig. 4.]*